# 08 -- Dynamic Trajectory Planning

Visualize a time-varying sunlight configuration space and the exact path that
waits for it to open. This notebook is based on
`examples/33_dynamic_trajectory.py` and compares both public CPU algorithms.

## Setup and interval occupancy

In [ ]:
import sys, os
from pathlib import Path

def _repo_root():
    """Find the Lunarscout repository root from the kernel working directory."""
    for start in [Path.cwd()] + list(Path.cwd().parents):
        if (start / "src" / "lunarscout" / "__init__.py").exists():
            return start
    raise RuntimeError(
        "Cannot locate Lunarscout repository root. "
        "Launch Jupyter from the repository root directory."
    )

_REPO = _repo_root()
sys.path.insert(0, str(_REPO / "src"))
sys.path.insert(0, str(_REPO / "examples"))


from datetime import datetime, timedelta, timezone

import lunarscout as ls
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

%matplotlib inline

from _example_support import synthetic_georef

georef = synthetic_georef(width=7, height=3, pixel_size=10.0, nodata=None)
start_time = datetime(2035, 1, 1, tzinfo=timezone.utc)
boundaries = tuple(start_time + timedelta(hours=i) for i in range(5))
sunlight = np.full((4, georef.height, georef.width), 255, dtype=np.uint8)
sunlight[:2, :, 4:] = 0
sunlight[:, 0, 3] = 0
start, goal = (0, 1), (6, 1)
allowed = sunlight >= round(0.2 * 255)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 5), constrained_layout=True)
for index, ax in enumerate(axes.flat):
    ax.imshow(allowed[index], cmap=ListedColormap(["#242424", "#f4d35e"]),
              vmin=0, vmax=1, origin="upper")
    ax.scatter(*start, color="cyan", edgecolor="black", s=70)
    ax.scatter(*goal, color="red", edgecolor="white", marker="*", s=120)
    ax.set_title(f"[{boundaries[index].hour:02d}:00, {boundaries[index+1].hour:02d}:00) UTC")
    ax.set(xticks=range(georef.width), yticks=range(georef.height))
fig.suptitle("Dynamic configuration space: yellow = allowed, dark = unavailable")
plt.show()

## Build the provider and run both exact algorithms

In [ ]:
signal = ls.trajectory.ArraySunlightProvider(boundaries, sunlight, georef)
configuration = ls.trajectory.AllOfConfigurationSpaceProvider((
    ls.trajectory.StaticConfigurationSpaceProvider(
        np.ones((georef.height, georef.width), dtype=bool), georef
    ),
    ls.trajectory.SunlightThresholdProvider(signal, 0.2),
))
model = ls.trajectory.StaticTravelModel(speed_m_per_h=20.0,
                                        include_diagonals=False)

results = {
    algorithm: ls.trajectory.dynamic_path(
        np.ones((georef.height, georef.width), dtype=bool),
        georef, start, goal, boundaries, configuration, start_time,
        model=model, algorithm=algorithm, backend="cpu",
    )
    for algorithm in ("gridrunner", "safe_interval")
}
for algorithm, result in results.items():
    print(f"{algorithm:13s}: arrival={result.arrival_time}, "
          f"elapsed={result.travel_time_hours:.2f} h, waits={result.wait_intervals}")

## Path, arrival time, and waiting location

In [ ]:
route = results["safe_interval"]
elapsed = np.array([
    (value - start_time).total_seconds() / 3600
    for value in route.arrival_times
])
fig, ax = plt.subplots(figsize=(11, 4))
ax.imshow(allowed[0], cmap=ListedColormap(["#242424", "#dddddd"]),
          vmin=0, vmax=1, origin="upper")
ax.plot(route.cells[:, 0], route.cells[:, 1], color="white", lw=3, zorder=2)
points = ax.scatter(route.cells[:, 0], route.cells[:, 1], c=elapsed,
                    cmap="plasma", edgecolor="black", s=110, zorder=3)
for index, hours in enumerate(elapsed):
    ax.annotate(f"{hours:.1f}h", route.cells[index] + np.array([0.08, -0.18]))
for wait_start, wait_stop in route.wait_intervals:
    wait_index = route.arrival_times.index(wait_start)
    ax.scatter(*route.cells[wait_index], marker="s", facecolors="none",
               edgecolors="cyan", linewidths=3, s=230, label="wait location")
ax.set(title="Dynamic path colored by arrival time", xlabel="x (cell)",
       ylabel="y (cell)", xticks=range(georef.width), yticks=range(georef.height))
fig.colorbar(points, ax=ax, label="hours after departure")
ax.legend()
fig.tight_layout()
plt.show()

## Route progression through configuration frames

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 5), constrained_layout=True)
for interval, ax in enumerate(axes.flat):
    ax.imshow(allowed[interval], cmap=ListedColormap(["#242424", "#f4d35e"]),
              vmin=0, vmax=1, origin="upper")
    arrived = np.array([
        value < boundaries[interval + 1] for value in route.arrival_times
    ])
    ax.plot(route.cells[:, 0], route.cells[:, 1], "--", color="white", alpha=0.5)
    ax.scatter(route.cells[arrived, 0], route.cells[arrived, 1],
               color="cyan", edgecolor="black", s=65)
    ax.set_title(f"By {boundaries[interval+1].hour:02d}:00 UTC")
    ax.set(xticks=range(georef.width), yticks=range(georef.height))
fig.suptitle("Cells reached by the end of each occupancy interval")
plt.show()